In [1]:
from tempfile import TemporaryDirectory

# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.datasets import (get_iris_dataset, get_letters_dataset,
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)
from utils.datasets import split_transpose

In [2]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()
y_full_encoded = label_encode(y_full)

In [3]:
QQDA = QDA()
TQDA = TensorizedQDA()

test_sz = 0.3
rng = 6663

# Generate current train-test split
X_train, X_test, y_train, y_test = split_transpose(
    X_full, y_full_encoded,
    test_size=test_sz,
    random_state=rng
)
TQDA.fit(X_train, y_train)
QQDA.fit(X_train, y_train)

In [4]:
print(f"Cantidad de matrices de covarianza: {len(QQDA.inv_covs)}")
print(f"Shape matriz 1: {QQDA.inv_covs[0].shape}")
print(f"Shape matriz 2: {QQDA.inv_covs[1].shape}")
print(f"Shape matriz 3: {QQDA.inv_covs[2].shape}")

Cantidad de matrices de covarianza: 3
Shape matriz 1: (13, 13)
Shape matriz 2: (13, 13)
Shape matriz 3: (13, 13)


In [5]:
print(f"Cantidad de matrices de medias: {len(QQDA.means)}")
print(f"Shape matriz 1: {QQDA.means[0].shape}")
print(f"Shape matriz 2: {QQDA.means[1].shape}")
print(f"Shape matriz 3: {QQDA.means[2].shape}")

Cantidad de matrices de medias: 3
Shape matriz 1: (13, 1)
Shape matriz 2: (13, 1)
Shape matriz 3: (13, 1)


In [6]:
tqda_predict =TQDA.predict(X_test)
print(tqda_predict)

[[0 0 1 1 1 2 0 0 2 1 2 0 0 2 2 1 0 0 1 0 0 0 1 1 1 1 1 1 1 1 0 1 1 2 1 1
  1 0 2 0 1 1 0 0 0 1 0 0 0 1 1 0 1 0]]


In [7]:
qda_predict = QQDA.predict(X_test)
print(qda_predict)

[[0 0 1 1 1 2 0 0 2 1 2 0 0 2 2 1 0 0 1 0 0 0 1 1 1 1 1 1 1 1 0 1 1 2 1 1
  1 0 2 0 1 1 0 0 0 1 0 0 0 1 1 0 1 0]]


In [8]:
print(TQDA.cc) # 54
print(QQDA.cc) # 54*3 = 162
# Nótese que desaparece el factor k
print(np.array_equal(tqda_predict, qda_predict))

54
162
True


## 1. ¿Sobre qué paraleliza `TensorizedQDA`?

¿Sobre las *k* clases, las *n* observaciones a predecir, o ambas?

`TensorizedQDA` paraleliza sobre las *k* clases. El shape de `TQDA.tensor_inv_cov` es `(3, 13, 13)`: el primer eje tiene una posición por cada una de las tres clases, y cada posición contiene una matriz de covarianza inversa de `13 × 13`.

## 2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

In [9]:
print(f"Shape del tensor matrices de covarianza de TensorizeQDA: {TQDA.tensor_inv_cov.shape}")
print(f"Shape del tensor medias de TensorizeQDA: {TQDA.tensor_means.shape}")


Shape del tensor matrices de covarianza de TensorizeQDA: (3, 13, 13)
Shape del tensor medias de TensorizeQDA: (3, 13, 1)


Como `X` está transpuesta, `m_obs = X.shape[1]` indica la cantidad `n` de observaciones. El `for` recorre esas `n` observaciones y construye un `y_hat` con una predicción para cada una.

El `for` de `predict` llama a `_predict_one` una vez por observación. `QDA` usa la implementación heredada, que recorre las clases con otro `for` y llama a `_predict_log_conditional`.
`TensorizedQDA` redefine `_predict_one` sin ese segundo `for` y llama a `_predict_log_conditionals`.

Si hay `n` observaciones y `k` clases, `_predict_log_conditionals` se ejecuta `n` veces en `TensorizedQDA` (una por observación), mientras que `_predict_log_conditional` se ejecuta `n × k` veces en `QDA` (una por cada combinación de observación y clase). Esto se debe a que `QDA` accede a los parámetros de cada clase por separado:

```python
inv_cov = self.inv_covs[class_idx]
unbiased_x = x - self.means[class_idx]
```

Si cada observación tiene `d` features, `tensor_means` tiene shape `(k, d, 1)` porque reúne las medias de las `k` clases, cada una de shape `(d, 1)`. En este ejemplo, esos shapes son `(3, 13, 1)` y `(13, 1)`, respectivamente. Al restarle `x`, de shape `(d, 1)`, se obtiene para cada clase la misma resta que `QDA` calcula con `x - self.means[class_idx]`. Del mismo modo, `tensor_inv_cov`, de shape `(k, d, d)`, reúne las mismas matrices inversas que `QDA` selecciona de `self.inv_covs`: el producto cuadrático y el término del determinante se calculan para cada clase, pero juntos en lugar de iterar sobre ellas. Así se obtiene un puntaje por clase; ambas implementaciones le suman los mismos logaritmos de las probabilidades a priori y eligen la clase con el mayor puntaje mediante `argmax`.